# NeuralAI-Air-135M Pre-Training (Colab T4 Optimized)

**Hardware:** Google Colab T4 GPU (16GB VRAM)
**Model:** 135M Llama (15 layers, 768 hidden, 32K vocab)
**Data:** 1B tokens from C4 + Books + OpenWebText + Wikipedia + StackExchange
**Duration:** ~60-90 days (nightly runs with Drive sync + resume)

**Critical:** This notebook auto-syncs checkpoints to Google Drive every 500 steps.
If Colab disconnects, re-run and it will resume from the latest Drive checkpoint.

In [ ]:
# === CELL 1: Setup & Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

# Create working dirs
WORK = Path('/content/neuralai')
WORK.mkdir(parents=True, exist_ok=True)
CKPT_LOCAL = WORK / 'checkpoints'
CKPT_DRIVE = Path('/content/drive/MyDrive/NeuralAI/checkpoints')
DATA_LOCAL = WORK / 'data'

CKPT_LOCAL.mkdir(parents=True, exist_ok=True)
CKPT_DRIVE.mkdir(parents=True, exist_ok=True)
DATA_LOCAL.mkdir(parents=True, exist_ok=True)

print(f'Work: {WORK}')
print(f'Checkpoints local: {CKPT_LOCAL}')
print(f'Checkpoints Drive: {CKPT_DRIVE}')
print(f'Data local: {DATA_LOCAL}')

# Install dependencies
!pip install -q transformers datasets accelerate safetensors tokenizers huggingface_hub
!pip install -q trl peft  # For SFT/DPO later

# Try flash-attn (T4 supports it)
try:
    !pip install -q flash-attn --no-build-isolation
    print('Flash Attention installed')
except Exception as e:
    print(f'Flash Attention install failed: {e}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# === CELL 2: Download model config & tokenizer ===
# Option A: If you uploaded NeuralAI-Air-135M-HF to Drive
DRIVE_MODEL = Path('/content/drive/MyDrive/NeuralAI-Air-135M-HF')
LOCAL_MODEL = WORK / 'NeuralAI-Air-135M-HF'

if DRIVE_MODEL.exists():
    !cp -r {DRIVE_MODEL} {LOCAL_MODEL}
    print('Copied from Drive')
else:
    # Option B: Clone from your repo (if public)
    !git clone https://github.com/Subject-Emu-5259/NeuralAI.git /tmp/neuralai_repo
    !cp -r /tmp/neuralai_repo/NeuralAI-Air-135M-HF {LOCAL_MODEL}
    print('Cloned from GitHub')

assert (LOCAL_MODEL / 'config.json').exists(), 'config.json missing!'
assert (LOCAL_MODEL / 'tokenizer.json').exists(), 'tokenizer.json missing!'
print('Model config & tokenizer ready')

# Verify architecture
import json
cfg = json.loads((LOCAL_MODEL / 'config.json').read_text())
print(f"Layers: {cfg['num_hidden_layers']}, Hidden: {cfg['hidden_size']}, Vocab: {cfg['vocab_size']}")
print(f"Heads: {cfg['num_attention_heads']}, KV Heads: {cfg['num_key_value_heads']}")


In [ ]:
# === CELL 3: Streaming Data Loader (no local storage) ===
# We stream from HuggingFace directly to avoid filling Colab disk
from datasets import load_dataset
from transformers import AutoTokenizer
import random

tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_MODEL))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = tokenizer.vocab_size
print(f'Tokenizer vocab: {VOCAB_SIZE}')

# Data source mix for streaming
# We download small chunks, tokenize on-the-fly, and discard raw text
SOURCES = [
    ('c4', 'en', 0.30, True),          # (name, config, weight, streaming)
    ('openwebtext', None, 0.20, False),
    ('wikitext', 'wikitext-103-raw-v1', 0.10, False),
    ('bookcorpus', None, 0.30, False),
    ('stacksample', None, 0.10, False),
]

def stream_tokens(source_name, source_config, split='train', max_docs=100000):
    """Yield token IDs from a dataset, one doc at a time."""
    ds = load_dataset(source_name, source_config, split=split, streaming=True)
    count = 0
    for doc in ds:
        if count >= max_docs:
            break
        text = doc.get('text', doc.get('content', str(doc)))
        if len(text) < 100:  # Skip tiny docs
            continue
        tokens = tokenizer.encode(text, truncation=True, max_length=512)
        if len(tokens) > 10:
            yield tokens
            count += 1

# Test stream
print('Testing data stream...')
test_stream = stream_tokens('c4', 'en', max_docs=100)
test_docs = [next(test_stream) for _ in range(5)]
print(f'Sample doc lengths: {[len(d) for d in test_docs]}')
print(f'Sample decoded: {tokenizer.decode(test_docs[0][:20])}')
print('Stream OK')


In [ ]:
# === CELL 4: Initialize Model (Random Weights) ===
from transformers import LlamaForCausalLM, LlamaConfig

config = LlamaConfig.from_pretrained(str(LOCAL_MODEL))

# T4-optimized: fp16 instead of bf16
model = LlamaForCausalLM(config).cuda().half()

# Enable gradient checkpointing (saves ~4GB VRAM)
model.gradient_checkpointing_enable()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded: {total/1e6:.1f}M params')
print(f'Trainable: {trainable/1e6:.1f}M')

# VRAM check
import torch
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f'VRAM allocated: {allocated:.1f} GB')
print(f'VRAM reserved: {reserved:.1f} GB')
print(f'Free: {16 - allocated:.1f} GB (of 16GB T4)')


In [ ]:
# === CELL 5: Training Arguments (T4-Optimized) ===
from transformers import TrainingArguments

# T4 constraints:
# - fp16 (not bf16)
# - batch 16 with grad accum 4 = effective 64
# - gradient checkpointing enabled
# - no torch.compile (T4 doesn't benefit much)

args = TrainingArguments(
    output_dir=str(CKPT_LOCAL),
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=6e-4,
    num_train_epochs=1,
    max_steps=30500,  # 1B tokens / (16*4*512) = ~30,500 steps
    warmup_steps=300,
    weight_decay=0.1,
    adam_beta1=0.9,
    adam_beta2=0.95,
    max_grad_norm=1.0,
    lr_scheduler_type='cosine',
    save_steps=500,
    save_total_limit=2,  # Only keep last 2 to save disk
    logging_steps=100,
    fp16=True,
    bf16=False,
    optim='adamw_torch',
    report_to='none',
    remove_unused_columns=False,
    dataloader_num_workers=0,  # Colab single process
    seed=42,
)

print('Training args configured for T4')
print(f'Effective batch: {args.per_device_train_batch_size * args.gradient_accumulation_steps}')
print(f'Context: 512, Total steps: {args.max_steps}')
print(f'LR: {args.learning_rate} → cosine decay')
print(f'Save every: {args.save_steps} steps')


In [ ]:
# === CELL 6: Resume Logic (Auto-detect Drive checkpoint) ===
"""
On Colab, sessions disconnect. We sync checkpoints to Drive and auto-resume.
"""

def find_latest_checkpoint():
    """Find latest checkpoint in Drive or local."""
    # Check Drive first (persisted across sessions)
    drive_checkpoints = sorted([d for d in CKPT_DRIVE.glob('checkpoint-*')])
    if drive_checkpoints:
        latest = drive_checkpoints[-1]
        print(f'Found Drive checkpoint: {latest.name}')
        return str(latest)
    
    # Check local (current session only)
    local_checkpoints = sorted([d for d in CKPT_LOCAL.glob('checkpoint-*')])
    if local_checkpoints:
        latest = local_checkpoints[-1]
        print(f'Found local checkpoint: {latest.name}')
        return str(latest)
    
    print('No checkpoint found. Starting from random init.')
    return None

resume_path = find_latest_checkpoint()


In [ ]:
# === CELL 7: Custom Dataset (Streaming + Packing) ===
from torch.utils.data import IterableDataset
import torch

class StreamingPackedDataset(IterableDataset):
    """Stream from HF datasets, pack into 512-token sequences."""
    def __init__(self, sources, tokenizer, context_length=512, seed=42):
        self.sources = sources
        self.tokenizer = tokenizer
        self.context_length = context_length
        self.seed = seed
    
    def __iter__(self):
        buffer = []
        worker_info = torch.utils.data.get_worker_info()
        
        # Shuffle source order per epoch
        rng = random.Random(self.seed + (worker_info.id if worker_info else 0))
        source_order = self.sources[:]
        rng.shuffle(source_order)
        
        for source_name, source_config, weight, streaming in source_order:
            try:
                if streaming:
                    ds = load_dataset(source_name, source_config, split='train', streaming=True)
                else:
                    ds = load_dataset(source_name, source_config, split='train')
                
                for doc in ds:
                    text = doc.get('text', doc.get('content', str(doc)))
                    if len(text) < 50:
                        continue
                    
                    tokens = self.tokenizer.encode(text, truncation=True, max_length=self.context_length)
                    if len(tokens) < 5:
                        continue
                    
                    buffer.extend(tokens)
                    buffer.append(self.tokenizer.eos_token_id)  # EOS between docs
                    
                    # Yield packed sequences
                    while len(buffer) >= self.context_length:
                        seq = buffer[:self.context_length]
                        buffer = buffer[self.context_length:]
                        yield {
                            'input_ids': torch.tensor(seq, dtype=torch.long),
                            'labels': torch.tensor(seq, dtype=torch.long),
                            'attention_mask': torch.ones(self.context_length, dtype=torch.long)
                        }
            except Exception as e:
                print(f'Source {source_name} failed: {e}. Skipping.')
                continue

# Create dataset
train_ds = StreamingPackedDataset(SOURCES, tokenizer, context_length=512, seed=42)
print('Streaming dataset ready')

# Test
it = iter(train_ds)
sample = next(it)
print(f'Sample shape: {sample["input_ids"].shape}')
print(f'Decoded: {tokenizer.decode(sample["input_ids"][:50])}')


In [ ]:
# === CELL 8: Trainer & Sync Callback ===
from transformers import Trainer, DataCollatorForLanguageModeling
from transformers.trainer_callback import TrainerCallback
import shutil

class DriveSyncCallback(TrainerCallback):
    """Sync checkpoints to Google Drive every N steps."""
    def __init__(self, local_dir, drive_dir, sync_every=500):
        self.local_dir = Path(local_dir)
        self.drive_dir = Path(drive_dir)
        self.sync_every = sync_every
        self.last_synced = -1
    
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % self.sync_every == 0:
            self._sync()
        return control
    
    def on_save(self, args, state, control, **kwargs):
        # Also sync when a checkpoint is saved
        self._sync()
        return control
    
    def _sync(self):
        checkpoints = sorted(self.local_dir.glob('checkpoint-*'))
        if not checkpoints:
            return
        
        latest = checkpoints[-1]
        step = int(latest.name.split('-')[-1])
        
        if step <= self.last_synced:
            return
        
        drive_ckpt = self.drive_dir / latest.name
        print(f'\n[DriveSync] Copying {latest.name} to Drive...')
        
        if drive_ckpt.exists():
            shutil.rmtree(drive_ckpt)
        shutil.copytree(latest, drive_ckpt)
        
        self.last_synced = step
        print(f'[DriveSync] Done. Checkpoint at step {step} synced.\n')

# Data collator (causal LM, no MLM)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    callbacks=[DriveSyncCallback(CKPT_LOCAL, CKPT_DRIVE, sync_every=500)]
)

print('Trainer ready with Drive sync')
if resume_path:
    print(f'Will resume from: {resume_path}')
else:
    print('Starting from random initialization')


In [ ]:
# === CELL 9: TRAIN ===
# This cell runs for hours. If Colab disconnects, re-run all cells above
# and this will auto-resume from the latest Drive checkpoint.

print('='*60)
print('STARTING PRE-TRAINING')
print('='*60)
print(f'Steps: {args.max_steps}')
print(f'Checkpoint dir: {CKPT_LOCAL}')
print(f'Drive sync dir: {CKPT_DRIVE}')
print(f'Resume from: {resume_path or "(random init)"}')
print('='*60)

trainer.train(resume_from_checkpoint=resume_path)

print('\nTraining complete!')
print(f'Final checkpoint: {CKPT_LOCAL}')
print('Run the next cell to copy final model to Drive.')


In [ ]:
# === CELL 10: Save Final to Drive ===
# Copy final model to Drive for permanent storage
FINAL_DIR = CKPT_DRIVE / 'final-pretrain'
FINAL_LOCAL = CKPT_LOCAL / 'final'

if FINAL_LOCAL.exists():
    if FINAL_DIR.exists():
        shutil.rmtree(FINAL_DIR)
    shutil.copytree(FINAL_LOCAL, FINAL_DIR)
    print(f'Final model saved to Drive: {FINAL_DIR}')
else:
    # Save last checkpoint as final
    checkpoints = sorted(CKPT_LOCAL.glob('checkpoint-*'))
    if checkpoints:
        last = checkpoints[-1]
        if FINAL_DIR.exists():
            shutil.rmtree(FINAL_DIR)
        shutil.copytree(last, FINAL_DIR)
        print(f'Last checkpoint copied to Drive: {FINAL_DIR}')
    else:
        print('No checkpoints found!')


## Colab Survival Guide

**If Colab disconnects (which it will):**
1. Re-run ALL cells above (Cells 1-8)
2. Cell 6 will auto-detect the latest checkpoint in Drive
3. Cell 9 resumes training from that checkpoint
4. You only lose the work between the last sync (every 500 steps)

**To prevent disconnects:**
- Keep the Colab tab active (don't minimize for hours)
- Use a browser extension that prevents idle timeout
- Run overnight with the tab pinned

**Expected timeline on T4:**
- ~500-800 tokens/sec (much slower than A100)
- 30,500 steps × 32,768 tokens = 1B tokens
- ~60-90 hours pure compute
- Realistically: 2-3 weeks with nightly runs and reconnects

**After pre-training completes:**
- Run SFT cell (load final checkpoint, train on v19 data)
- Run DPO cell (load SFT checkpoint, train on v19 preferences)
- Export to GGUF and deploy
